## Part 1a: Loading a dataset from HuggingFace and Transforming with a transformer-based tokenizer

In [ ]:
#import necessary modules
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error() 


import warnings
warnings.filterwarnings("ignore")


from ner_pipeline.preprocessing.iob_converter import SpacyIOBConverter, HFIOBConverter
from ner_pipeline.schemas.ner_params import IOBConfig, RawNerSchema, nlp

/nfs/production/literature/amina-mardiyyah/envs/test-env/lib/python3.11/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
#load dataset from hf_hub. Here we load an example from OTAR3088 hf repository
dataset = load_dataset("OTAR3088/CellFinder-all")
dataset

In [ ]:
#select relevant columns ['sentences', 'entities']
dataset = load_dataset("OTAR3088/CellFinder-all", columns=["sentences","entities"])
dataset

In [ ]:
#check the structure of th entities col as we need that to define our config schema
dataset["train"][0]

In [15]:
#define the dataset schema
data_schema = {"text_col": "sentences", "label_col": "entities", "ent_label_key":"label"}

In [9]:
#load a biomedical tokenizer
ckpt = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
tokenizer = AutoTokenizer.from_pretrained(ckpt)
tokenizer

BertTokenizerFast(name_or_path='microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext', vocab_size=30522, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [16]:
#initialise the IOBConfig class
config = IOBConfig(schema = data_schema,
                   tokenizer_backend = tokenizer,
                   as_hf_dataset=True #whether to return a hf dataset or not
                   )
config

IOBConfig(schema=RawNerSchema(text_col='sentences', label_col='entities', ent_label_key='label'), tokenizer_backend=BertTokenizerFast(name_or_path='microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext', vocab_size=30522, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, sin

In [17]:
#intialise the HFIOBConverter and parse dataset
converter = HFIOBConverter(dataset, config)
iob_dataset_hf = converter.convert()


Map:   0%|          | 0/2219 [00:00<?, ? examples/s]

In [18]:
#inspect sample after conversion
iob_dataset_hf["train"][0]

{'tokens': ['human',
  'embryonic',
  'stem',
  'cells',
  '(',
  'hescs',
  ')',
  'and',
  'neural',
  'progenitor',
  '(',
  'np',
  ')',
  'cells',
  'are',
  'excellent',
  'models',
  'for',
  'recapit',
  '##ulating',
  'early',
  'neuronal',
  'development',
  'in',
  'vitro',
  ',',
  'and',
  'are',
  'key',
  'to',
  'establishing',
  'strategies',
  'for',
  'the',
  'treatment',
  'of',
  'degenerative',
  'disorders',
  '.'],
 'tags': ['B-CellType',
  'I-CellType',
  'I-CellType',
  'I-CellType',
  'O',
  'B-CellType',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O']}

## Part 1B: Use a Spacy tokenizer

In [19]:
#reintialise config with spacy backend
config = IOBConfig(schema = data_schema,
                   tokenizer_backend = nlp,
                   as_hf_dataset=True #whether to return a hf dataset or not
                   )
config

IOBConfig(schema=RawNerSchema(text_col='sentences', label_col='entities', ent_label_key='label'), tokenizer_backend=<spacy.lang.en.English object at 0x7fa33e9fe210>, as_hf_dataset=True)

In [20]:
converter = SpacyIOBConverter(dataset, config)
iob_dataset_spacy = converter.convert()

#inspect sample after conversion
iob_dataset_spacy["train"][0]

Map:   0%|          | 0/2219 [00:00<?, ? examples/s]

{'tokens': ['Human',
  'embryonic',
  'stem',
  'cells',
  '(',
  'hESCs',
  ')',
  'and',
  'neural',
  'progenitor',
  '(',
  'NP',
  ')',
  'cells',
  'are',
  'excellent',
  'models',
  'for',
  'recapitulating',
  'early',
  'neuronal',
  'development',
  'in',
  'vitro',
  ',',
  'and',
  'are',
  'key',
  'to',
  'establishing',
  'strategies',
  'for',
  'the',
  'treatment',
  'of',
  'degenerative',
  'disorders',
  '.'],
 'tags': ['B-CellType',
  'I-CellType',
  'I-CellType',
  'I-CellType',
  'O',
  'B-CellType',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O']}

In [ ]:
#you can push dataset to hub or save to a pandas dataset. To push to hub, run the following cell
#change repo_id to desired name
iob_dataset_hf.push_to_hub("Sample_IOB_Dataset", 
                           private=True #whether to push dataset as private or public
                          )

In [26]:
#save to a dataframe
import pandas as pd
iob_dataset_df = pd.DataFrame(iob_dataset_spacy["train"])
iob_dataset_df.head()

,tokens,tags
0,"[Human, embryonic, stem, cells, (, hESCs, ), a...","[B-CellType, I-CellType, I-CellType, I-CellTyp..."
1,"[While, much, effort, had, been, undertaken, t...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,"[Alternative, RNA, splicing, (, AS, ), ,, a, m...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,"[Human, ESC, ,, hESC-derived, NP, ,, and, huma...","[B-CellType, I-CellType, O, O, O, O, O, B-Cell..."
4,"[We, introduced, an, outlier, detection, appro...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
